# Retail dbt Test Generation Pipeline

End-to-end, fully automated pipeline: PostgreSQL retail schema → column profiling → Ollama LLM → `dbt-expectations` tests → dbt run → results dashboard.

## Pipeline Architecture

```
PostgreSQL  ──►  Schema Extraction  ──►  Column Profiling
                      │                        │
                      ▼                        ▼
              dbt_profiler Models      LLM Test Generation
              (auto-generated)         (dbt-expectations)
                      │                        │
                      └──────────┬─────────────┘
                                 ▼
                        Assemble schema.yml
                        + staging SQL models
                                 │
                                 ▼
                          dbt run / test
                                 │
                                 ▼
                        Results Dashboard
```

## Packages Used
| Package | Purpose |
|---------|---------|
| `dbt-labs/dbt_utils` | Expression tests, utility macros |
| `metaplane/dbt_expectations` | Rich column-level data quality tests |
| `data-mie/dbt_profiler` | Automated column profiling — null rates, uniqueness, min/max, std deviation |
| `Ollama (local LLM)` | Interprets profiler stats → generates `dbt_expectations` test configs |

> **Nothing is hardcoded.** Tables, columns, tests, and schema files are all derived dynamically from the live PostgreSQL schema.


## 1. Environment Setup

Clears stale environment variables and reloads the `.env` file.

**Why this cell exists:** Jupyter kernels cache env vars between runs. Without the `pop()` calls, a stale `POSTGRES_HOST` from a previous session would silently override the `.env` reload.

**Variables loaded from `.env`:**
- `POSTGRES_HOST / PORT / USER / PASSWORD / DB / SCHEMA`
- `OLLAMA_HOST / PORT / MODEL`
- `DBT_PROJECT_DIR` — resolved later in the connectivity cell


In [1]:
# Needed to clear the .env variabel path

import os

from dotenv import load_dotenv

os.environ.pop("POST_TEST", None)
os.environ.pop("POSTGRES_HOST", None)

load_dotenv("../.env", override=True)


True

## 2. Connectivity Check & Config

Reads all service config from env vars and verifies both PostgreSQL and Ollama are reachable before the pipeline proceeds.

**How it works:**
1. Reads connection params via `os.getenv()` with safe defaults
2. Attempts `psycopg2.connect()` to PostgreSQL — closes immediately after success
3. HTTP GET `http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags` — expects status 200
4. Prints a pass/fail summary per service

**Outputs set:** `PG_HOST`, `PG_PORT`, `PG_USER`, `PG_PASSWORD`, `PG_DB`, `PG_SCHEMA`, `OLLAMA_HOST`, `OLLAMA_PORT`, `OLLAMA_MODEL`, `DBT_PROJECT_DIR`


In [2]:
# Cell 1: Setup & Connectivity Check
import os
import psycopg2
import requests

from dotenv import load_dotenv

load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

# Config from env
PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", "5435"))
PG_USER = os.getenv("POSTGRES_USER", "retail")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "retail123")
PG_DB = os.getenv("POSTGRES_DB", "retail")
PG_SCHEMA = os.getenv("POSTGRES_SCHEMA", "public")

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost")
OLLAMA_PORT = int(os.getenv("OLLAMA_PORT", "11434"))
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:0.5b")

DBT_PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "dbt_project"))

# Connectivity checks
checks = {}

# PostgreSQL
try:
    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        user=PG_USER,
        password=PG_PASSWORD,
        dbname=PG_DB,
        connect_timeout=5,
    )
    conn.close()
    checks["PostgreSQL"] = True
except Exception as e:
    checks["PostgreSQL"] = str(e)

# Ollama
try:
    r = requests.get(f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags", timeout=5)
    checks["Ollama"] = r.status_code == 200
except Exception as e:
    checks["Ollama"] = str(e)

print("=== Connectivity Checks ===")
for service, status in checks.items():
    icon = "OK" if status is True else "FAIL"
    print(f"  [{icon}] {service}: {status if status is not True else 'connected'}")


=== Connectivity Checks ===
  [OK] PostgreSQL: connected
  [OK] Ollama: connected


## 3. Table Discovery

Discovers all tables in the configured PostgreSQL schema using `pipeline.dbt_generator.list_tables_with_columns`.

**How it works:**
- Queries `information_schema.columns` for all tables in `PG_SCHEMA`
- Returns a list of `{name, fqn, columns[]}` dicts — one entry per table
- Adds project root to `sys.path` so `pipeline.*` imports resolve correctly

**Output:** `table_metadata` — list of table dicts, each with column name and basic type info


In [3]:
# Cell 2: Load Table Metadata from PostgreSQL
import sys
from pathlib import Path

package_root = Path.cwd().parent
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from pipeline.dbt_generator import list_tables_with_columns


def load_table_metadata():
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
    )
    try:
        return list_tables_with_columns(conn, schema=PG_SCHEMA)
    finally:
        conn.close()


table_metadata = load_table_metadata()
print(
    f"Loaded metadata for {len(table_metadata)} tables from PostgreSQL schema '{PG_SCHEMA}'."
)


Loaded metadata for 14 tables from PostgreSQL schema 'public'.


## 4. Schema Extraction (Rich Metadata)

Enriches raw table discovery with full column metadata needed for LLM prompting and test generation.

**How it works:**
- Single SQL query joins `information_schema.columns`, `table_constraints` (PK), and `key_column_usage` (FK)
- Normalises nullable flags, data types (e.g. `character varying(255)` → `varchar(255)`), and FK references
- Builds two Pandas DataFrames:
  - `schema_df` — one row per table with column counts, PK/FK counts, distinct type counts
  - `columns_df` — one row per column with full metadata including `ordinal_position`, `is_primary_key`, `is_foreign_key`, `references`

**Outputs set:** `schema_df`, `columns_df`, `table_columns`, `table_names`, `table_schemas`

> `columns_df` is the primary input for all downstream LLM prompts and file generation — it drives everything without manual table/column lists.


In [4]:
# Cell 3: Extract Schema from PostgreSQL
# Build rich table/column metadata for downstream LLM prompting
import psycopg2
import pandas as pd


def _pick(d, *keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def _to_bool_nullable(v):
    if isinstance(v, bool):
        return v
    if v is None:
        return None
    return str(v).strip().lower() in {"yes", "true", "t", "1"}


def fetch_information_schema_map(schema_name, table_list):
    if not table_list:
        return {}

    query = """
    WITH cols AS (
        SELECT
            c.table_schema,
            c.table_name,
            c.column_name,
            c.ordinal_position,
            c.is_nullable,
            c.column_default,
            c.data_type,
            c.udt_name,
            c.character_maximum_length,
            c.numeric_precision,
            c.numeric_scale
        FROM information_schema.columns c
        WHERE c.table_schema = %s
          AND c.table_name = ANY(%s)
    ),
    pk AS (
        SELECT kcu.table_schema, kcu.table_name, kcu.column_name
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema = kcu.table_schema
         AND tc.table_name = kcu.table_name
        WHERE tc.constraint_type = 'PRIMARY KEY'
          AND tc.table_schema = %s
          AND tc.table_name = ANY(%s)
    ),
    fk AS (
        SELECT
            kcu.table_schema,
            kcu.table_name,
            kcu.column_name,
            ccu.table_name AS foreign_table,
            ccu.column_name AS foreign_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema = kcu.table_schema
         AND tc.table_name = kcu.table_name
        JOIN information_schema.constraint_column_usage ccu
          ON ccu.constraint_name = tc.constraint_name
         AND ccu.table_schema = tc.table_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = %s
          AND tc.table_name = ANY(%s)
    )
    SELECT
        cols.table_name,
        cols.column_name,
        cols.ordinal_position,
        cols.is_nullable,
        cols.column_default,
        cols.data_type,
        cols.udt_name,
        cols.character_maximum_length,
        cols.numeric_precision,
        cols.numeric_scale,
        (pk.column_name IS NOT NULL) AS is_primary_key,
        (fk.column_name IS NOT NULL) AS is_foreign_key,
        fk.foreign_table,
        fk.foreign_column
    FROM cols
    LEFT JOIN pk
      ON pk.table_schema = cols.table_schema
     AND pk.table_name = cols.table_name
     AND pk.column_name = cols.column_name
    LEFT JOIN fk
      ON fk.table_schema = cols.table_schema
     AND fk.table_name = cols.table_name
     AND fk.column_name = cols.column_name
    ORDER BY cols.table_name, cols.ordinal_position;
    """

    info_map = {}
    with psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        connect_timeout=5,
    ) as conn2:
        with conn2.cursor() as cur:
            cur.execute(
                query,
                (
                    PG_SCHEMA,
                    table_list,
                    PG_SCHEMA,
                    table_list,
                    PG_SCHEMA,
                    table_list,
                ),
            )
            rows = cur.fetchall()

    for row in rows:
        (
            tname,
            cname,
            ordinal_position,
            is_nullable,
            column_default,
            data_type,
            udt_name,
            char_len,
            num_precision,
            num_scale,
            is_pk,
            is_fk,
            foreign_table,
            foreign_column,
        ) = row

        # Better human-readable datatype
        if data_type == "character varying" and char_len:
            dtype = f"varchar({char_len})"
        elif data_type == "character" and char_len:
            dtype = f"char({char_len})"
        elif data_type == "numeric" and num_precision is not None:
            dtype = (
                f"numeric({num_precision},{num_scale})"
                if num_scale is not None
                else f"numeric({num_precision})"
            )
        else:
            dtype = data_type or udt_name or "unknown"

        info_map[(tname, cname)] = {
            "ordinal_position": ordinal_position,
            "nullable": _to_bool_nullable(is_nullable),
            "default": column_default,
            "data_type": dtype,
            "is_primary_key": bool(is_pk),
            "is_foreign_key": bool(is_fk),
            "references": (
                f"{foreign_table}.{foreign_column}"
                if foreign_table and foreign_column
                else None
            ),
            "character_maximum_length": char_len,
            "numeric_precision": num_precision,
            "numeric_scale": num_scale,
        }

    return info_map


if table_metadata:
    table_rows = []
    column_rows = []
    table_schemas = {}

    table_names = [t["name"] for t in table_metadata]
    info_schema_map = fetch_information_schema_map(PG_SCHEMA, table_names)

    for t in table_metadata:
        table_name = t["name"]
        fqn = t.get("fqn", table_name)
        cols = t.get("columns", []) or []

        table_schemas[table_name] = {
            "name": table_name,
            "fqn": fqn,
            "columns": cols,
        }

        data_types = []
        nullable_count = 0
        pk_count = 0
        fk_count = 0

        for idx, c in enumerate(cols, start=1):
            col_name = _pick(c, "name", "column_name", default=f"col_{idx}")
            meta = info_schema_map.get((table_name, col_name), {})

            data_type = meta.get(
                "data_type",
                _pick(
                    c, "data_type", "dataType", "type", "udt_name", default="unknown"
                ),
            )
            nullable = meta.get(
                "nullable",
                _to_bool_nullable(_pick(c, "is_nullable", "nullable", default=None)),
            )
            default_val = meta.get(
                "default", _pick(c, "column_default", "default", default=None)
            )
            is_pk = meta.get(
                "is_primary_key",
                bool(_pick(c, "is_primary_key", "primary_key", "pk", default=False)),
            )
            is_fk = meta.get(
                "is_foreign_key",
                bool(_pick(c, "is_foreign_key", "foreign_key", "fk", default=False)),
            )

            if nullable is True:
                nullable_count += 1
            if is_pk:
                pk_count += 1
            if is_fk:
                fk_count += 1

            data_types.append(str(data_type))

            column_rows.append(
                {
                    "table_name": table_name,
                    "fqn": fqn,
                    "column_name": col_name,
                    "ordinal_position": meta.get(
                        "ordinal_position",
                        _pick(c, "ordinal_position", "position", default=idx),
                    ),
                    "data_type": data_type,
                    "nullable": nullable,
                    "default": default_val,
                    "is_primary_key": is_pk,
                    "is_foreign_key": is_fk,
                    "references": meta.get(
                        "references",
                        _pick(
                            c,
                            "references",
                            "foreign_table",
                            "foreign_column",
                            default=None,
                        ),
                    ),
                    "comment": _pick(c, "comment", "description", default=None),
                    "character_maximum_length": meta.get(
                        "character_maximum_length",
                        _pick(
                            c, "character_maximum_length", "max_length", default=None
                        ),
                    ),
                    "numeric_precision": meta.get(
                        "numeric_precision", _pick(c, "numeric_precision", default=None)
                    ),
                    "numeric_scale": meta.get(
                        "numeric_scale", _pick(c, "numeric_scale", default=None)
                    ),
                }
            )

        table_rows.append(
            {
                "table_name": table_name,
                "fqn": fqn,
                "column_count": len(cols),
                "nullable_columns": nullable_count,
                "primary_key_columns": pk_count,
                "foreign_key_columns": fk_count,
                "distinct_data_types": len(set(data_types)),
                "data_types": ", ".join(sorted(set(data_types))),
            }
        )

    schema_df = (
        pd.DataFrame(table_rows).sort_values("table_name").reset_index(drop=True)
    )
    columns_df = (
        pd.DataFrame(column_rows)
        .sort_values(["table_name", "ordinal_position"])
        .reset_index(drop=True)
    )

    print("Tables extracted from PostgreSQL:")
    display(schema_df)

    print("Column-level schema details:")
    display(columns_df)

    table_columns = {t["name"]: t["columns"] for t in table_metadata}
    table_names = [t["name"] for t in table_metadata]

else:
    print(
        "No tables found. Make sure seed.py ran and the PostgreSQL schema is populated."
    )
    table_columns = {}
    table_names = []
    table_schemas = {}
    schema_df = pd.DataFrame()
    columns_df = pd.DataFrame()


Tables extracted from PostgreSQL:


,table_name,fqn,column_count,nullable_columns,primary_key_columns,foreign_key_columns,distinct_data_types,data_types
0,customers,public.customers,8,2,1,0,6,"boolean, integer, timestamp without time zone,..."
1,employees,public.employees,8,1,1,2,5,"date, integer, numeric(10,2), varchar(100), va..."
2,inventory_snapshots,public.inventory_snapshots,6,0,1,2,2,"date, integer"
3,loyalty_points,public.loyalty_points,6,0,1,1,2,"integer, timestamp without time zone"
4,order_items,public.order_items,6,0,1,2,2,"integer, numeric(10,2)"
5,orders,public.orders,11,1,1,3,5,"integer, numeric(10,2), timestamp without time..."
6,product_categories,public.product_categories,4,2,1,1,3,"integer, text, varchar(100)"
7,product_reviews,public.product_reviews,7,2,1,2,4,"integer, text, timestamp without time zone, va..."
8,products,public.products,9,1,1,2,5,"boolean, integer, numeric(10,2), varchar(200),..."
9,promotions,public.promotions,7,0,1,0,6,"boolean, integer, numeric(10,2), timestamp wit..."


Column-level schema details:


,table_name,fqn,column_name,ordinal_position,data_type,nullable,default,is_primary_key,is_foreign_key,references,comment,character_maximum_length,numeric_precision,numeric_scale
0,customers,public.customers,customer_id,1,integer,False,nextval('customers_customer_id_seq'::regclass),True,False,NaN,None,NaN,32.0,0.0
1,customers,public.customers,first_name,2,varchar(100),False,NaN,False,False,NaN,None,100.0,NaN,NaN
2,customers,public.customers,last_name,3,varchar(100),False,NaN,False,False,NaN,None,100.0,NaN,NaN
3,customers,public.customers,email,4,varchar(100),True,NaN,False,False,NaN,None,100.0,NaN,NaN
4,customers,public.customers,phone,5,varchar(30),True,NaN,False,False,NaN,None,30.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,suppliers,public.suppliers,contact_name,3,varchar(100),True,NaN,False,False,NaN,None,100.0,NaN,NaN
99,suppliers,public.suppliers,email,4,varchar(100),True,NaN,False,False,NaN,None,100.0,NaN,NaN
100,suppliers,public.suppliers,phone,5,varchar(30),True,NaN,False,False,NaN,None,30.0,NaN,NaN
101,suppliers,public.suppliers,country,6,varchar(100),False,NaN,False,False,NaN,None,100.0,NaN,NaN


## 5. Sample Data Preview *(Optional — currently disabled)*

Fetches and displays up to 5 rows per table as a quick sanity check on data content.

**Status:** Commented out to keep pipeline execution lean. Uncomment to inspect raw data during development.

**How it works:** `pipeline.dbt_generator.get_sample_rows()` runs `SELECT * FROM {table} LIMIT n` per table and renders via `display(pd.DataFrame(rows))`.


In [5]:
# # Cell 4: Sample Data Preview
# from pipeline.dbt_generator import get_sample_rows

# conn = psycopg2.connect(
#     host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
# )

# print("Sample data (5 rows per table):")
# for table in table_names[:5]:  # preview first 5 tables
#     rows = get_sample_rows(conn, table, limit=5)
#     if rows:
#         print(f"\n--- {table} ---")
#         display(pd.DataFrame(rows))


## 6. Write Staging Models & Generate Profiling SQL

Writes staging SQL views and `sources.yml` to the dbt project — **no `schema.yml` yet** (that needs LLM output).
Also generates `models/profiling/profile_stg_{table}.sql` per table.

**Why here:** Staging views must exist in the database before `dbt_profiler` can query them.

**Files written:**
- `models/staging/sources.yml`
- `models/staging/stg_{table}.sql` — `select * from source(...)` view per table
- `models/profiling/profile_stg_{table}.sql` — `dbt_profiler.profile(ref(...))` per table
- `models/profiling/schema.yml`


In [6]:
import os
import yaml
from pipeline.dbt_generator import write_staging_models, write_profiling_models

print("Writing staging models...")
write_staging_models(DBT_PROJECT_DIR, table_names, source_name="retail")

print("\nWriting profiling models...")
write_profiling_models(DBT_PROJECT_DIR, table_names)

print(
    f"\nDone: {len(table_names)} staging + {len(table_names)} profiling models written."
)


Writing staging models...
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/sources.yml
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_customers.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_employees.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_inventory_snapshots.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_loyalty_points.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_order_items.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_orders.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg_product_categories.sql
  Written: /home/fm-pc-lt-314/All_Repo/datalens/retail-dbt-poc/dbt_project/models/staging/stg

## 7. Run dbt — Staging + Profiling

Materialises staging views then runs `dbt_profiler` — must happen BEFORE LLM so profiler stats are in the DB.

| Command | What it does |
|---------|-------------|
| `dbt deps` | Installs `dbt_utils`, `dbt_expectations`, `dbt_profiler` |
| `dbt run --select staging` | Creates `stg_{table}` views from PostgreSQL sources |
| `dbt run --select profiling` | Runs `dbt_profiler.profile()` → stores per-column metrics in `profiling.profile_stg_{table}` |


In [ ]:
import subprocess


def run_dbt_cmd(args, cwd):
    cmd = ["dbt"] + args + ["--profiles-dir", "."]
    print(f"Running: dbt {' '.join(args)}")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr and result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    return result.returncode


print("=== dbt deps ===")
run_dbt_cmd(["deps"], DBT_PROJECT_DIR)

print("\n=== dbt run — staging ===")
rc_staging = run_dbt_cmd(["run", "--select", "staging"], DBT_PROJECT_DIR)

if rc_staging == 0:
    print("\n=== dbt run — profiling (dbt_profiler) ===")
    rc_profiling = run_dbt_cmd(["run", "--select", "profiling"], DBT_PROJECT_DIR)
    if rc_profiling != 0:
        print("WARNING: profiling run failed — profiler stats will be empty")
else:
    print("\nERROR: staging failed — skipping profiling")


=== dbt deps ===
Running: dbt deps
04:20:38  Running with dbt=1.11.11
04:20:40  Installing dbt-labs/dbt_utils
04:20:42  Installed from version 1.3.3
04:20:42  Up to date!
04:20:42  Installing metaplane/dbt_expectations
04:20:43  Installed from version 0.10.10
04:20:43  Up to date!
04:20:43  Installing data-mie/dbt_profiler
04:20:44  Installed from version 1.0.0
04:20:44  Up to date!
04:20:44  Installing godatadriven/dbt_date
04:20:45  Installed from version 0.18.0
04:20:45  Up to date!


=== dbt run — staging ===
Running: dbt run --select staging


## 8. Load Column Stats from dbt_profiler

Queries `profiling.profile_stg_{table}` tables populated by the previous cell.
Builds `all_stats = {table: {col: {metric: value}}}` — used by dashboard charts (cell 13).

**dbt_profiler metrics used:** `not_null_proportion`, `distinct_proportion`, `is_unique`, `min`, `max`, `avg`, `std_deviation`, `median`

> Stats are NOT passed to the LLM. LLM test generation (cell 9) uses schema metadata only. Stats are available for the dashboard visualisation.


In [ ]:
import psycopg2
import pandas as pd
from pipeline.dbt_generator import load_profiler_stats

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
)

all_stats = load_profiler_stats(conn, table_names, profiling_schema="profiling")

missing = [t for t in table_names if not all_stats.get(t)]
if missing:
    print(f"WARNING: dbt_profiler tables missing for: {missing}")
    print("Make sure 'dbt run --select profiling' completed successfully.")
else:
    print(f"Stats loaded from dbt_profiler for {len(all_stats)} tables.")

# Build stats_df — all raw + derived metrics, no predefined column filter
stats_rows = []
for table, col_stats in all_stats.items():
    for col_name, s in col_stats.items():
        stats_rows.append({"table": table, "column": col_name, **s})
stats_df = pd.DataFrame(stats_rows) if stats_rows else pd.DataFrame()

if not stats_df.empty:
    priority = [
        "table",
        "column",
        "data_type",
        "row_count",
        "null_count",
        "null_rate",
        "null_proportion",
        "not_null_proportion",
        "distinct_count",
        "distinct_proportion",
        "uniqueness_score",
        "is_unique",
        "min_val",
        "max_val",
        "mean",
        "median",
        "std_dev_population",
        "std_dev_sample",
        "cv",
        "profiled_at",
    ]
    ordered = [c for c in priority if c in stats_df.columns]
    extras = [c for c in stats_df.columns if c not in set(ordered)]
    display(stats_df[ordered + extras])
    print(
        f"\n{len(stats_df)} column profiles | {len(stats_df.columns) - 2} metrics per column"
    )
else:
    print("No profiler stats available yet.")


## 9. LLM-Driven Per-Column Test Generation


In [ ]:
import os
import json
import re
import yaml
import pandas as pd
from pipeline.llm_client import (
    OllamaClient,
    build_column_prompt,
    build_table_dict_prompt,
    parse_column_output,
    fill_test_configs,
)
from pipeline.dbt_generator import get_sample_rows

EXPECT_DIR = os.path.join(DBT_PROJECT_DIR, "expectations")
DICT_DIR = os.path.join(EXPECT_DIR, "data_dictionary")
os.makedirs(EXPECT_DIR, exist_ok=True)
os.makedirs(DICT_DIR, exist_ok=True)

ollama = OllamaClient(OLLAMA_HOST, OLLAMA_PORT, OLLAMA_MODEL)
if not ollama.health():
    raise RuntimeError(f"Ollama not reachable at {OLLAMA_HOST}:{OLLAMA_PORT}")

generated_tests = {}  # {table: {col: [test_objects]}}


def _slug(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()


def _col_sample_values(table: str, col: str, limit: int = 5):
    try:
        rows = get_sample_rows(conn, table, limit=limit)
    except Exception:
        rows = []
    return [
        None if (v := r.get(col)) is None else str(v)
        for r in rows
        if isinstance(r, dict) and col in r
    ][:limit]


def _tests_to_yaml(col: str, tests: list[dict]) -> str:
    col_block = {"name": col, "tests": []}
    for t in tests:
        name = t.get("test", "")
        cfg = t.get("config") or {}
        col_block["tests"].append(name if not cfg else {name: cfg})
    return yaml.dump([col_block], default_flow_style=False, allow_unicode=True)


for table in table_names:
    print(f"\n=== Processing table: {table} ===")
    table_cols = columns_df[columns_df["table_name"] == table].sort_values(
        "ordinal_position"
    )
    if table_cols.empty:
        continue

    table_column_context = [
        {
            "name": r["column_name"],
            "dtype": str(r.get("data_type") or "unknown"),
            "nullable": bool(r.get("nullable", True)),
            "is_primary_key": bool(r.get("is_primary_key", False)),
            "is_foreign_key": bool(r.get("is_foreign_key", False)),
            "references": r.get("references"),
        }
        for _, r in table_cols.iterrows()
    ]

    # Step 1: Generate table-level data dictionary (schema + samples, one LLM call per table)
    samples_by_col = {
        r["column_name"]: _col_sample_values(table, r["column_name"])
        for _, r in table_cols.iterrows()
    }
    dict_prompt = build_table_dict_prompt(table, table_column_context, samples_by_col)
    dict_raw = ollama.generate(dict_prompt)
    dict_parsed = parse_column_output(dict_raw)

    table_dict = None
    if dict_parsed and isinstance(dict_parsed, dict):
        dict_path = os.path.join(DICT_DIR, f"{_slug(table)}__dict.json")
        with open(dict_path, "w", encoding="utf-8") as df:
            json.dump(dict_parsed, df, indent=2, default=str)
        table_dict = dict_parsed
        print(f"  Data dictionary written: {dict_path}")
    else:
        raw_dict_path = os.path.join(DICT_DIR, f"{_slug(table)}__dict_raw.txt")
        with open(raw_dict_path, "w", encoding="utf-8") as rf:
            rf.write(dict_raw)
        print(f"  Data dictionary parse failed — raw saved")

    # Step 2: Per-column test generation
    # LLM receives: schema (col types/PK/FK), full table data dictionary, per-column entry
    for _, crow in table_cols.iterrows():
        col = crow["column_name"]
        dtype = str(crow.get("data_type", "") or "")
        nullable = bool(crow.get("nullable", True))
        is_primary_key = bool(crow.get("is_primary_key", False))
        is_foreign_key = bool(crow.get("is_foreign_key", False))
        references = crow.get("references")

        col_dict = (table_dict.get("columns") or {}).get(col) if table_dict else None

        prompt = build_column_prompt(
            table=table,
            col=col,
            dtype=dtype,
            nullable=nullable,
            is_primary_key=is_primary_key,
            is_foreign_key=is_foreign_key,
            references=references,
            table_columns=table_column_context,  # all cols: name/dtype/PK/FK
            data_dictionary=col_dict,  # this column's dict entry
            table_data_dictionary=table_dict,  # full table dict: all col descriptions + semantics
        )

        raw = ollama.generate(prompt)
        parsed = parse_column_output(raw)
        base_name = f"expect_{_slug(table)}__{_slug(col)}"

        if parsed and isinstance(parsed, dict):
            tests = parsed.get("tests") or []
            tests = fill_test_configs(col, dtype)(tests)
            generated_tests.setdefault(table, {})[col] = tests
            rationale = parsed.get("rationale", "")

            if tests:
                yaml_path = os.path.join(EXPECT_DIR, f"{base_name}__tests.yml")
                with open(yaml_path, "w", encoding="utf-8") as yf:
                    yf.write(f"# dbt-expectations tests for {table}.{col}\n")
                    yf.write(f"# {rationale}\n")
                    yf.write(_tests_to_yaml(col, tests))

            meta_path = os.path.join(EXPECT_DIR, f"{base_name}__meta.json")
            with open(meta_path, "w", encoding="utf-8") as mf:
                json.dump(
                    {
                        "table": table,
                        "column": col,
                        "dtype": dtype,
                        "nullable": nullable,
                        "is_primary_key": is_primary_key,
                        "is_foreign_key": is_foreign_key,
                        "references": references,
                        "tests_generated": len(tests),
                        "rationale": rationale,
                    },
                    mf,
                    default=str,
                    indent=2,
                )
            print(f"  {col}: {len(tests)} tests")
        else:
            raw_path = os.path.join(EXPECT_DIR, f"{base_name}__raw.txt")
            with open(raw_path, "w", encoding="utf-8") as rf:
                rf.write(raw)
            print(f"  {col}: LLM parse failed — raw saved")

print(f"\nDone. Test YAML files: {EXPECT_DIR}")


---

> **Cells below (11–12) are deprecated reference implementations.** They are commented out and not part of the active pipeline.

## 10. Assemble & Write dbt Files


In [ ]:
# Cell 10: Write Generated Files to dbt Project (+ dbt_expectations for all columns)
import yaml
from pipeline.dbt_generator import write_schema_yml


def _normalized_dbt_type(dtype: str) -> str:
    d = (dtype or "").lower()
    if "int" in d or "serial" in d:
        return "integer"
    if "numeric" in d or "decimal" in d:
        return "numeric"
    if "double" in d or "float" in d or d == "real":
        return "float"
    if "bool" in d:
        return "boolean"
    if "date" in d and "time" not in d:
        return "date"
    if "time" in d:
        return "timestamp"
    if any(tok in d for tok in ["char", "text", "uuid", "json", "jsonb"]):
        return "text"
    return "text"


def build_schema_from_metadata() -> str:
    """
    Build models/staging/schema.yml from:
    - table_names (from DB)
    - columns_df (types, nullable, PK/FK from information_schema)
    - generated_tests (LLM-generated dbt-expectations tests from cell 9)
    No profiler stats — baselines use schema semantics only.
    """
    models = []
    # for table in table_names:
    for table in table_names[1]:  # only first table for testing
        model_name = f"stg_{table}"
        cols_meta = columns_df[columns_df["table_name"] == table].sort_values(
            "ordinal_position"
        )
        columns_out = []
        for _, row in cols_meta.iterrows():
            col_name = row["column_name"]
            expected_type = _normalized_dbt_type(str(row.get("data_type", "text")))
            nullable = bool(row.get("nullable", True))
            is_pk = bool(row.get("is_primary_key", False))

            tests = []
            existing = set()

            # Merge LLM-generated dbt-expectations tests
            llm_tests = generated_tests.get(table, {}).get(col_name, [])
            for t in llm_tests:
                name = t.get("test", "")
                cfg = t.get("config") or {}
                if name and name not in existing:
                    tests.append(name if not cfg else {name: cfg})
                    existing.add(name)

            # Baseline: type check (schema-derived)
            type_test = "dbt_expectations.expect_column_values_to_be_of_type"
            if type_test not in existing:
                tests.append({type_test: {"column_type": expected_type}})
                existing.add(type_test)

            # Baseline: not_null if schema says NOT NULL and LLM missed it
            nn_test = "dbt_expectations.expect_column_values_to_not_be_null"
            if not nullable and nn_test not in existing:
                tests.append(nn_test)
                existing.add(nn_test)

            # Baseline: unique for primary keys if LLM missed it
            uq_test = "dbt_expectations.expect_column_values_to_be_unique"
            if is_pk and uq_test not in existing:
                tests.append(uq_test)
                existing.add(uq_test)

            columns_out.append(
                {
                    "name": col_name,
                    "description": "",
                    "tests": tests,
                }
            )

        models.append(
            {
                "name": model_name,
                "description": f"Staging model for {table}",
                "columns": columns_out,
            }
        )

    doc = {"version": 2, "models": models}
    return yaml.dump(doc, default_flow_style=False, sort_keys=False, allow_unicode=True)


merged_yml = build_schema_from_metadata()

print(f"Writing schema.yml to: {DBT_PROJECT_DIR}")
write_schema_yml(DBT_PROJECT_DIR, merged_yml)
print("Done — schema.yml written with LLM-generated dbt_expectations tests.")


## 11. Run dbt — Tests Only


In [ ]:
# Run dbt test — staging + profiling already ran in cell 7
print("=== dbt test (with store-failures) ===")
run_dbt_cmd(["test", "--store-failures", "--select", "staging"], DBT_PROJECT_DIR)

print("\n=== dbt docs generate ===")
run_dbt_cmd(["docs", "generate"], DBT_PROJECT_DIR)


## 12. Test Results Dashboard


In [ ]:
# Cell 9: Results Dashboard
import json

run_results_path = os.path.join(DBT_PROJECT_DIR, "target", "run_results.json")

if not os.path.exists(run_results_path):
    print("No run_results.json found. Run Cell 8 first.")
else:
    with open(run_results_path) as f:
        run_results = json.load(f)

    results = run_results.get("results", [])
    rows = []
    for r in results:
        node = r.get("unique_id", "")
        node_name = node.split(".")[-1]
        rows.append(
            {
                "test_name": node_name,
                "status": r.get("status", "unknown"),
                "execution_time_s": round(r.get("execution_time", 0), 2),
                "failures": r.get("failures", 0),
                "message": (r.get("message") or "")[:80],
            }
        )

    results_df = pd.DataFrame(rows).sort_values(
        ["status", "failures"], ascending=[True, False]
    )

    total = len(results_df)
    passed = (results_df["status"] == "pass").sum()
    failed = (results_df["status"] == "fail").sum()

    print(f"=== dbt Test Results ===")
    print(f"Total: {total} | Passed: {passed} | Failed: {failed}")
    print()
    display(results_df)


## 13. Column Metrics Visualization


In [ ]:
# Cell 10: Column Metrics Chart
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.figsize"] = (12, 5)

# Chart 1: Test results by status
if "results_df" in dir() and not results_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    status_counts = results_df["status"].value_counts()
    colors = {
        "pass": "#4CAF50",
        "fail": "#F44336",
        "warn": "#FF9800",
        "error": "#9C27B0",
    }
    bar_colors = [colors.get(s, "#999") for s in status_counts.index]
    ax1.bar(status_counts.index, status_counts.values, color=bar_colors)
    ax1.set_title("dbt Tests by Status")
    ax1.set_xlabel("Status")
    ax1.set_ylabel("Count")
    for i, (idx, val) in enumerate(status_counts.items()):
        ax1.text(i, val + 0.1, str(val), ha="center", va="bottom", fontweight="bold")

    # Chart 2: Top 10 columns by null_rate
    if not stats_df.empty:
        top_nulls = stats_df[stats_df["null_rate"] > 0].nlargest(10, "null_rate")
        if not top_nulls.empty:
            labels = top_nulls["table"] + "." + top_nulls["column"]
            ax2.barh(labels, top_nulls["null_rate"] * 100, color="#F44336", alpha=0.7)
            ax2.set_title("Top 10 Columns by Null Rate")
            ax2.set_xlabel("Null Rate (%)")
            ax2.invert_yaxis()
        else:
            ax2.text(
                0.5,
                0.5,
                "No nulls found!",
                ha="center",
                va="center",
                transform=ax2.transAxes,
            )
            ax2.set_title("Null Rates")

    plt.tight_layout()
    plt.savefig(
        os.path.join("..", "dbt_test_results.png"), dpi=100, bbox_inches="tight"
    )
    plt.show()
    print("Chart saved to dbt_test_results.png")


## 14. Pipeline Summary & Documentation


In [ ]:
# Cell 11: Pipeline Summary & Links
import subprocess

# Start dbt docs serve in background
docs_proc = subprocess.Popen(
    ["dbt", "docs", "serve", "--port", "8082", "--profiles-dir", "."],
    cwd=DBT_PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("=== Pipeline Complete ===")
print(f"Tables discovered from PostgreSQL : {len(table_names)}")
print(
    f"Columns profiled                  : {len(stats_rows) if 'stats_rows' in dir() else 'N/A'}"
)
print(
    f"LLM tests generated (tables)      : {len(generated_tests) if 'generated_tests' in dir() else 'N/A'}"
)
col_test_count = (
    sum(len(v) for t in generated_tests.values() for v in t.values())
    if "generated_tests" in dir()
    else 0
)
print(f"LLM tests generated (columns)     : {col_test_count}")

if "results_df" in dir() and not results_df.empty:
    passed = (results_df["status"] == "pass").sum()
    failed = (results_df["status"] == "fail").sum()
    total = len(results_df)
    print(
        f"dbt tests                         : {total} total | {passed} passed | {failed} failed"
    )

print()
print("Links:")
print(f"  dbt Docs   →  http://localhost:8082")
print(f"  PostgreSQL →  localhost:{PG_PORT}")
print(f"  Ollama     →  http://localhost:{OLLAMA_PORT}")
print()
print("To stop dbt docs server: docs_proc.terminate()")

if "conn" in dir() and conn and not conn.closed:
    conn.close()
